Transformers 라이브러리

인코더 기반 모델: 예시 모델 Multiple Choice
- 문제 예시) 여러개 입력이 주어졌을때 주어진 문장중 옳은 문장을 고르는 객관식 문제, 
- 입력 프롬프트 한개와 그에 대응하는 n개 후보로 구성됨
- 프롬프트 + 답변1, 프롬프트 + 답변2,...프롬프트 + 답변n 목록을 모두 입력 받아 후보중 무엇이 정답인지 출력

1. klue/bert-base 기본 모델: BertForMultipleChoice
- 추론 : 수능 국어 문제 풀이 - 5지선다형(보기 5개)

In [2]:
# 인코더 기반 모델
# 1. klue/bert-base 기본 모델: BertForMultipleChoice
# 예를 들어, "지문"과 "보기 1, 2, 3, 4, 5"가 주어졌을 때 각각의 보기가 정답일 확률(점수)을 계산하여 
# 가장 높은 점수를 가진 보기를 선택할 수 있게 만들어 준다
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice

# 모델 이름 지정 및 사전학습 모델 로드
model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMultipleChoice.from_pretrained(model_name)

# 모델 출력 확인
print(f"model : {model}")

/Users/ai/deep-learning/LLM/rag_pdf_qdrant/venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertForMultipleChoice were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model : BertForMultipleChoice(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [48]:
# 인코더 기반 모델
# 1. klue/bert-base 기본 모델: BertForMultipleChoice
# 추론 : 수능 국어 문제 풀이 - 5지선다형(보기 5개)
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice

# 1. 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# 2. 모델 이름 지정 및 사전학습 모델 로드
model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMultipleChoice.from_pretrained(model_name)
print(f"model: {model}")

# 3. 모델 -> 디바이스 이동
model.to(device)

# 4. 평가/추론시 설정: 드롭아웃 비활성화, 정규화 레이어 고정
model.eval()

# 5. 수능 지문 및 5지선다 보기 설정
passage = "다음 글을 읽고 물음에 답하시오. (중략) 화자는 흐르는 강물을 바라보며 세월의 무상함과 인생의 덧없음을 달래고 있다."
choices = [
    "① 현실의 타협과 물질적 성취를 추구하고 있다.",
    "② 자연 친화적 삶을 통해 내면의 평온을 찾고 있다.",
    "③ 세월의 흐름에 따른 삶의 덧없음을 달래고 있다.",
    "④ 문명의 이기를 비판하며 도시로 회귀하고 있다.",
    "⑤ 과거의 역사적 사건을 객관적으로 고증하고 있다."
]

# 6. 토크나이징 (질문 + 답변 결합)
# - choices 리스트에는 1~5번까지의 보기(답변)가 들어 있음
# - 지문과 결합하는 작업을 5번 반복, 예) 1번째 반복 지문 + 1번 보기, 2번째 반복 지문 + 2번 보기 ...
encoded_inputs = [
    # text=passage, text_pair=choice 질문과 답변의 결합
    # 허깅페이스 토크나이저에서 text/text_pair 인자를 받으면 두문장을 하나의 문장처럼 이어 붙이면서 중간과 끝에 특수 토큰[CLS] [SEP] 삽입해준다
    # 예) [CLS] 지문 내용... [SEP] 3번 보기 내용... [SEP]
    tokenizer(
        text=passage,
        text_pair=choice,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128,
    # 리스트 컴프리헨션(List Comprehension)이 실행되고 나면, encoded_inputs라는 리스트 안에
    # 총 5개의 토큰화된 결과 묶음이 들어가게 된다.
    ) for choice in choices
]

# 7. 3차원 텐서 구성 및 디바이스 리동
# 5개의 보기 텐서를 하나로 묶고([batch_size,num_choices,sequence] 예)[1,5,128]), 모델과 같은 디바이스로 보낸다
# 3차원 텐서 구성 작업 설명
# - .squeeze(0): [1,128] 불필요한 첫번째 차원(1)을 제거하여 -> [128] 모양의 1차원 텐서로 만들어준다
# - torch.stack(tensors=..., dim=0): 리스트안에 5개의 텐서[128]를 새로운 차원(dim=0)을 기준으로 수직으로 쌓아서 하나의 큰 텐서로 합쳐준다
# - 예)[128] 1차원 텐서 -> [5,128] 2차원 텐서로 만든다, 여기서 5는 보기 개수(num_choices=5)가 된다
# - .unsqueeze(0): 텐서의 맨 앞에 크기가 1인 차원(dim=0)을 새롭게 끼워 넣는다
# - 예)[5,128] 2차원 텐서 -> [1,5,128] 3차원 텐서로 만든다, 여기서 1은 배치 크기(batch_size=1)가 된다, 지금 한번에 처리할 문항(배치)이 1개 있다
# - .to(device): [1,5,128] 최종 3차원 텐서 -> 디비이스 이동
input_ids = torch.stack(
    tensors=[item['input_ids'].squeeze(0) for item in encoded_inputs],
    dim=0,
).unsqueeze(0).to(device)
print(f"input_ids: {input_ids}")
print(f"input_ids shape: {input_ids.shape}")

attention_mask = torch.stack(
    tensors=[item['attention_mask'].squeeze(0) for item in encoded_inputs],
    dim=0,
).unsqueeze(0).to(device)
print(f"attention_mask: {attention_mask}")
print(f"attention_mask shape: {attention_mask.shape}")

# 8. 모델 추론
with torch.no_grad(): # 미분 사용하지 않음
    # 모델 추론 전체 결과
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    )
    print(f"outputs: {outputs}")

    # logits 결과: [1,5] (보기 5개 각각의 점수)
    logits = outputs.logits
    print(f"logits: {logits}, {logits.shape}")

# 9. 결과 확인 및 정답 예측
# - torch.softmax(): logits [1,5] (batch_size, num_choices) 소프트맥스 함수는 어떤 축(Dimension)을 기준으로 확률의 합이 1이 되도록 만들것인가를 결정해야 된다
# - 예)5개의 보기(1~5번) 각각의 정답일 확률을 계산해서 그 5개의 확률 합이 1.0(100%) 되게 한다
# - .squeeze(): logits 모양 [1,5] 2차원 텐서 -> [5] 1차원 텐서로 만든다, 불필요한 차원을 제거
# - .tolist(): 텐서 -> 리스트 자료형으로 최종 만든다
# 소프트맥스 계산
probs = torch.softmax(
    input=logits,
    dim=-1
).squeeze().tolist()
print(f"probs: {probs}")

# 로짓값: 차원축소,리스트 형변환
scores = logits.squeeze().tolist()
print(f"scores: {scores}")

# 가장 높은 로짓값의 인덱스 번호
predicted_idx = torch.argmax(
    input=logits,
    dim=-1,
).item()
print(f"predicted_idx: {predicted_idx}")

print("\n=== 5지선다 추론 결과 ===")
for idx, (choice, score, prob) in enumerate(zip(choices, scores, probs)):
    print(f"{choice} -> 점수(logits): {score:.4f} | 확률: {prob * 100:.4f}%")

print(f"\n모델이 선택한 정답: {predicted_idx + 1}번 ({choices[predicted_idx]})")

Device: cpu


/Users/ai/deep-learning/LLM/rag_pdf_qdrant/venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertForMultipleChoice were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model: BertForMultipleChoice(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-